### Split RNA multiome by cell types

In [32]:
import os
import numpy as np
import pandas as pd
import scanpy as sc 
from collections import Counter

### Split for all multiome adata

In [2]:
multiome_adata = sc.read_h5ad("02_combined_multiome_RNA.h5ad")

In [12]:
multiome_adata.shape

(304969, 16115)

In [5]:
multiome_ATAC_adata = sc.read_h5ad("02_combined_multiome_ATAC.h5ad")

In [13]:
multiome_ATAC_adata.shape

(304969, 606219)

In [4]:
Counter(multiome_adata.obs.final_cell_type)

Counter({'Cardiomyocyte': 117639,
         'Fibroblast': 75263,
         'Myeloid': 35630,
         'Pericyte': 32395,
         'Endothelial': 17685,
         'Lymphoid': 10244,
         'vSMC': 6395,
         'Endocardial': 3352,
         'Neuronal': 2823,
         'Mast': 1092,
         'Epicardial': 1070,
         'LEC': 984,
         'Adipocyte': 397})

In [10]:
multiome_ATAC_adata.obs_names.equals(multiome_adata.obs_names)

True

In [7]:
Counter(multiome_ATAC_adata.obs.cell_type)

Counter({'Cardiomyocyte': 117827,
         'Fibroblast': 75649,
         'Myeloid': 35040,
         'Pericyte': 33685,
         'Endothelial': 23043,
         'Lymphoid': 10647,
         'vSMC': 4939,
         'Neuronal': 2129,
         'Epicardial': 1047,
         'Mast': 963})

In [17]:
# Cast to string to avoid categorical comparison error
cell_types = multiome_ATAC_adata.obs["cell_type"].astype(str)
final_cell_types = multiome_adata.obs["final_cell_type"].astype(str)

# Boolean comparison
comparison = cell_types == final_cell_types

# Show only mismatches
mismatches = pd.DataFrame({
    "cell_type": cell_types[~comparison],
    "final_cell_type": final_cell_types[~comparison]
})

print(mismatches.head())
print(f"Total mismatches: {len(mismatches)}")

                                                        cell_type  \
multiome_barcode                                                    
HCAHeart9845431_HCAHeart9917173:CCCTTAATCTATTGTC-1       Pericyte   
HCAHeartST10773165_HCAHeartST10781062:GTCAAACTC...  Cardiomyocyte   
HCAHeartST10773165_HCAHeartST10781062:TATCGCACA...     Fibroblast   
HCAHeartST11064575_HCAHeartST11023240:AGGTTAACA...        Myeloid   
HCAHeartST10773166_HCAHeartST10781063:GCAAGCCTC...       Lymphoid   

                                                   final_cell_type  
multiome_barcode                                                    
HCAHeart9845431_HCAHeart9917173:CCCTTAATCTATTGTC-1            vSMC  
HCAHeartST10773165_HCAHeartST10781062:GTCAAACTC...         Myeloid  
HCAHeartST10773165_HCAHeartST10781062:TATCGCACA...       Adipocyte  
HCAHeartST11064575_HCAHeartST11023240:AGGTTAACA...            Mast  
HCAHeartST10773166_HCAHeartST10781063:GCAAGCCTC...         Myeloid  
Total mismatches: 10846


In [18]:
mismatches

,cell_type,final_cell_type
multiome_barcode,,
HCAHeart9845431_HCAHeart9917173:CCCTTAATCTATTGTC-1,Pericyte,vSMC
HCAHeartST10773165_HCAHeartST10781062:GTCAAACTCCGCTAGA-1,Cardiomyocyte,Myeloid
HCAHeartST10773165_HCAHeartST10781062:TATCGCACAATCTCTC-1,Fibroblast,Adipocyte
HCAHeartST11064575_HCAHeartST11023240:AGGTTAACAGGCTGTT-1,Myeloid,Mast
HCAHeartST10773166_HCAHeartST10781063:GCAAGCCTCTCCTCTT-1,Lymphoid,Myeloid
...,...,...
ENCODE v4 (Snyder):ENCFF802AQC:AGCTATATCTTAGTGA,Endothelial,Endocardial
ENCODE v4 (Snyder):ENCFF776DQR:TTTACGCGTGGGTGAA,Endothelial,Endocardial
ENCODE v4 (Snyder):ENCSR762LML:GATTCAGGTGCTCCGT,Fibroblast,vSMC


In [24]:
filt_multiome_RNA_adata = multiome_adata[~multiome_adata.obs_names.isin(mismatches.index), :]

In [28]:
Counter(filt_multiome_RNA_adata.obs.final_cell_type)

Counter({'Cardiomyocyte': 117622,
         'Fibroblast': 74129,
         'Myeloid': 34564,
         'Pericyte': 31692,
         'Endothelial': 17609,
         'Lymphoid': 10048,
         'vSMC': 4475,
         'Neuronal': 2115,
         'Mast': 935,
         'Epicardial': 934})

In [25]:
filt_multiome_ATAC_adata = multiome_ATAC_adata[~multiome_ATAC_adata.obs_names.isin(mismatches.index), :]

In [27]:
print(filt_multiome_RNA_adata.shape)
print(filt_multiome_ATAC_adata.shape)

(294123, 16115)
(294123, 606219)


#### For scE2G computation, filter to just 10K cells per cell type max

In [33]:
def subsample_adata(adata, groupby_col="final_cell_type", n_cells=10000, random_state=0):
    
    np.random.seed(random_state)
    
    keep_indices = []
    for group, idx in adata.obs.groupby(groupby_col).indices.items():
        idx = np.array(list(idx))
        if len(idx) > n_cells:
            chosen = np.random.choice(idx, size=n_cells, replace=False)
        else:
            chosen = idx
        keep_indices.extend(chosen)
    
    return adata[keep_indices].copy()

In [34]:
filt_multiome_RNA_adata_sub = subsample_adata(filt_multiome_RNA_adata, "final_cell_type", n_cells=10000)

/mnt/data1/william/tmp/ipykernel_1343869/233015639.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for group, idx in adata.obs.groupby(groupby_col).indices.items():


In [35]:
filt_multiome_ATAC_adata_sub = filt_multiome_ATAC_adata[filt_multiome_RNA_adata_sub.obs_names]

In [37]:
print(filt_multiome_ATAC_adata_sub.shape)
print(filt_multiome_RNA_adata_sub.shape)

(68459, 606219)
(68459, 16115)


In [39]:
Counter(filt_multiome_ATAC_adata_sub.obs.cell_type)

Counter({'Cardiomyocyte': 10000,
         'Endothelial': 10000,
         'Fibroblast': 10000,
         'Lymphoid': 10000,
         'Myeloid': 10000,
         'Pericyte': 10000,
         'vSMC': 4475,
         'Neuronal': 2115,
         'Mast': 935,
         'Epicardial': 934})

In [41]:
# save the subsampled ATAC
filt_multiome_ATAC_adata_sub.write("03_subsampled_ATAC.h5ad")

In [45]:
os.makedirs("subsampled_multiome_RNA/", exist_ok=True)

In [43]:
multiome_cell_types = filt_multiome_RNA_adata_sub.obs.final_cell_type.unique()

### scE2G recommends at least 2 million fragments, 1 million UMIs 

Check which cell types have at least 1 million UMIs

In [47]:
%%time
num_UMIs_per_cell_type = list()

for cell_type in multiome_cell_types:
    filtered_adata = filt_multiome_RNA_adata_sub[filt_multiome_RNA_adata_sub.obs.final_cell_type == cell_type].copy()
    num_UMIs = filtered_adata.X.sum()
    num_UMIs_per_cell_type.append(num_UMIs)
    filtered_adata.write("subsampled_multiome_RNA/" + cell_type + ".h5ad")

CPU times: user 3.05 s, sys: 3.79 s, total: 6.84 s
Wall time: 6.87 s


In [55]:
num_UMI_df = pd.DataFrame({'cell_type': multiome_cell_types,
                         'total_UMI': num_UMIs_per_cell_type})

In [61]:
num_UMI_df[num_UMI_df['total_UMI'] < 1000000]

,cell_type,total_UMI
5,Mast,995443.0


From the RNA standpoint, we will perform this for all cells except for Mast cells